In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Memuat Dataset Klasifikasi Kanker
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

print(f"Dimensi Dataset: {X.shape}")
print(f"Distribusi Kelas (0 = Ganas, 1 = Jinak):\n{pd.Series(y).value_counts()}")

# 2. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scaling Fitur (Sangat penting untuk Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Inisialisasi Model
model_logreg = LogisticRegression(max_iter=10000, random_state=42)
model_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Melatih Model menggunakan Data Latih
model_logreg.fit(X_train_scaled, y_train)
model_dt.fit(X_train, y_train)  # Tree-based model tidak wajib scaled data
model_rf.fit(X_train, y_train)

print("Proses pelatihan 3 model selesai!")

In [ ]:
from sklearn.metrics import classification_report

# Prediksi Data Uji
y_pred_logreg = model_logreg.predict(X_test_scaled)
y_pred_dt = model_dt.predict(X_test)
y_pred_rf = model_rf.predict(X_test)

print("========== LOGISTIC REGRESSION REPORT ==========")
print(classification_report(y_test, y_pred_logreg, target_names=cancer.target_names))

print("\n========== DECISION TREE REPORT ==========")
print(classification_report(y_test, y_pred_dt, target_names=cancer.target_names))

print("\n========== RANDOM FOREST REPORT ==========")
print(classification_report(y_test, y_pred_rf, target_names=cancer.target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

models_preds = [
    ("Logistic Regression", y_pred_logreg),
    ("Decision Tree", y_pred_dt),
    ("Random Forest", y_pred_rf)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_name, y_pred) in zip(axes, models_preds):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=cancer.target_names, yticklabels=cancer.target_names)
    ax.set_title(f'Confusion Matrix:\n{model_name}')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Mendapatkan probabilitas prediksi untuk kelas positif
prob_logreg = model_logreg.predict_proba(X_test_scaled)[:, 1]
prob_dt = model_dt.predict_proba(X_test)[:, 1]
prob_rf = model_rf.predict_proba(X_test)[:, 1]

# Hitung ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, prob_logreg)
fpr_dt, tpr_dt, _ = roc_curve(y_test, prob_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_test, prob_rf)

# Plotting Kurva ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, prob_logreg):.3f})')
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {roc_auc_score(y_test, prob_dt):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, prob_rf):.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## Kesimpulan dan Analisis Model Comparison

Berdasarkan hasil eksperimen klasifikasi pada dataset Wisconsin Breast Cancer menggunakan framework Scikit-Learn, diperoleh analisis sebagai berikut:

1. **Performa Model Terbaik:** Algoritma **Random Forest Classifier** memberikan nilai F1-Score dan Area Under Curve (AUC) tertinggi mendekati sempurna. Hal ini disebabkan karena sifat ensemble-nya yang menggabungkan banyak pohon keputusan acak, sehingga mampu mereduksi varians dan mencegah overfitting dengan sangat baik dibandingkan single Decision Tree.
2. **Analisis Sensitivitas Fitur:** Algoritma **Logistic Regression** membutuhkan tahapan normalisasi data (`StandardScaler`) agar nilai bobot koefisiennya konvergen. Performa model linear ini cukup bersaing karena karakteristik dataset kanker payudara memiliki pola pemisahan linier yang cukup kuat pada ruang dimensi tinggi.
3. **Kelemahan Single Model:** **Decision Tree** menunjukkan skor evaluasi yang paling rendah di antara ketiganya karena rentan mengalami masalah overfitting pada kedalaman pohon tertentu, sehingga akurasi prediksi pada data uji menurun.